# 🖥️ Sampling a Field
The particle trajectories allow us to study fields like temperature, plastic concentration or chlorophyll from a Lagrangian perspective.

In this tutorial we will go through how particles can sample a {py:obj}`parcels.Field`, using temperature as an example. Along the way we will get to know the {py:obj}`parcels.Variable` class and some of its methods. This tutorial covers several applications of a sampling setup:

- [**Basic along trajectory sampling**](#basic-sampling)
- [**Sampling initial Field values**](#sampling-initial-field-values)
- [**Sampling velocity fields**](#sampling-velocity-fields)

## Basic sampling

We import both the packages that we need to set up the simulation, as well as the Parcels package.

In [ ]:
# Modules needed for the Parcels simulation
from datetime import timedelta

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import parcels
import parcels.tutorial

Suppose we want to study the environmental temperature for plankton drifting in the Agulhas current. We have a CopernicusMarine dataset with surface ocean velocities and the corresponding potential temperature ("thetao") stored in netcdf files in the [Parcels example dataset repository](https://github.com/Parcels-code/parcels-data). Loading in the FieldSet, Parcels detects U and V because they have CF standard names and tells us that they are assumed as the velocity fields to be used in the simulation.


In [ ]:
# Load the CopernicusMarine data in the Agulhas region from the example_datasets
ds_fields = parcels.tutorial.open_dataset(
    "CopernicusMarine_data_for_Argo_tutorial/data"
)

# Convert to SGRID-compliant dataset and create FieldSet
fields = {"U": ds_fields["uo"], "V": ds_fields["vo"], "thetao": ds_fields["thetao"]}
ds_fset = parcels.convert.copernicusmarine_to_sgrid(fields=fields)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

# Convert the FieldSet to windowed arrays for better performance
fieldset = fieldset.to_windowed_arrays()

 Ten particles are initialized at the surface in the centre of our domain, at the initial time step.

In [ ]:
# Particle locations and initial time
npart = 10  # number of particles to be released
lon = 32 * np.ones(npart)
lat = np.linspace(-32.5, -30.5, npart, dtype=np.float32)
time = np.repeat(
    ds_fields.time.values[0], npart
)  # release all particles at the start time of the fieldset
z = np.repeat(0.5, npart)  # upper level is at 0.5m depth

# Plot temperature field and initial particle locations
plt.figure()
ax = plt.axes()
T_contour = ax.contourf(
    ds_fields.longitude.values,
    ds_fields.latitude.values,
    ds_fields.thetao.values[0, 0],
    cmap=plt.cm.inferno,
    vmin=20,
    vmax=27,
)
ax.scatter(lon, lat, c="w")
plt.colorbar(T_contour, label=r"T [$^{\circ} C$]")
plt.show()

To sample the temperature field, we need to create a new class of particles where temperature is a `Variable`. We then also need a new Kernel `SampleT` that interpolates the temperature field at the particle location and stores that in `particle.temperature`.


In [ ]:
SampleParticle = parcels.Particle.add_variable(
    parcels.Variable("temperature", dtype=np.float32, initial=np.nan)
)


def SampleT(particles, fieldset):
    particles.temperature = fieldset.thetao[
        particles.t, particles.z, particles.y, particles.x
    ]

We can then sample and advect by combining the `SampleT` and `AdvectionRK2` kernels in a list. Note that the order does not matter.


In [ ]:
pset = parcels.ParticleSet(
    fieldset=fieldset, pclass=SampleParticle, x=lon, y=lat, z=z, t=time
)

output_file = parcels.ParticleFile("sampletemp.parquet", outputdt=timedelta(hours=1))

pset.execute(
    [parcels.kernels.AdvectionRK2, SampleT],
    runtime=timedelta(hours=30),
    dt=timedelta(minutes=5),
    output_file=output_file,
)

The particle dataset now contains the particle trajectories and the corresponding environmental temperature


In [ ]:
df = parcels.read_particlefile("sampletemp.parquet")

plt.figure()
ax = plt.axes()
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
for traj in df.partition_by("particle_id", maintain_order=True):
    ax.plot(traj["x"], traj["y"], c="k", zorder=1)
    T_scatter = ax.scatter(
        traj["x"],
        traj["y"],
        c=traj["temperature"],
        cmap=plt.cm.inferno,
        norm=mpl.colors.Normalize(vmin=22.0, vmax=24.0),
        edgecolor="k",
        zorder=2,
    )
plt.colorbar(T_scatter, label=r"T [$^{\circ} C$]")
plt.show()

## Sampling initial Field values

When creating a ParticleSet, the initial values of `Variables` are set as a constant (default `0`, but in the example above set to `np.nan`). Those will also be the values that appear as first entry in the ParticleFile, as seen when printing the initial values:

In [ ]:
print(
    [
        traj["temperature"][0]
        for traj in df.partition_by("particle_id", maintain_order=True)
    ]
)

If you want to sample the initial value of a field, you can do so by calling the field with the ParticleSet as an argument. For example, in, the code above you would add one line (`pset.temperature = fieldset.thetao[pset]`) to sample the initial temperature values, before calling `pset.execute(...)`.

In [ ]:
pset = parcels.ParticleSet(
    fieldset=fieldset, pclass=SampleParticle, x=lon, y=lat, z=z, t=time
)
pset.temperature = fieldset.thetao[pset]  # <-- sample initial temperature values

output_file = parcels.ParticleFile(
    "sampletemp_withinit.parquet", outputdt=timedelta(hours=1)
)

pset.execute(
    [parcels.kernels.AdvectionRK2, SampleT],
    runtime=timedelta(hours=30),
    dt=timedelta(minutes=5),
    output_file=output_file,
)

In [ ]:
df = parcels.read_particlefile("sampletemp_withinit.parquet")
print(
    [
        round(traj["temperature"][0], 2)
        for traj in df.partition_by("particle_id", maintain_order=True)
    ]
)

```{note}
Note that you need to explicitly set the `t` values in the ParticleSet; otherwise you will get a `ValueError: Time values cannot be NaN.` error. This is because (if not specified) the time values are NaN until they get set on `pset.execute()`, when the sign of `dt` is known.
```

## Sampling velocity fields
Because Parcels works also for generalised curvilinear grids, you need to tread somewhat carefully when wanting to sample the velocity fields `U` and `V`. In fact, Parcels will throw a warning when directly calling a sampling of either of these fields:

In [ ]:
def SampleVel_wrong(particles, fieldset):
    u = fieldset.U[particles]


pset = parcels.ParticleSet(
    fieldset=fieldset, pclass=parcels.Particle, x=lon, y=lat, z=z, t=time
)

pset.execute(
    SampleVel_wrong,
    runtime=timedelta(hours=30),
    dt=timedelta(minutes=5),
)

Instead, you should use the code `u, v = fieldset.UV[...]`. With this code, the sampling is consistent with the actual velocity fields used in the advection Kernels. The difference is that on a curvilinear grid, `fieldset.U[..]` returns the velocity in the `i`-direction (the columns on the grid), while `fieldset.UV[...]` returns the velocities in the longitude and latitude direction - see also the [velocity conversion tutorial](./tutorial_velocityconversion.ipynb). Furthermore, only `fieldset.UV[...]` sampling can correctly deal with boundary conditions such as `freeslip` and `partialslip` ([tutorial_unstuck_Agrid](tutorial_unstuck_Agrid.ipynb))


In [ ]:
def SampleVel_correct(particles, fieldset):
    u, v = fieldset.UV[particles]


pset = parcels.ParticleSet(
    fieldset=fieldset, pclass=parcels.Particle, x=lon, y=lat, z=z, t=time
)

pset.execute(
    SampleVel_correct,
    runtime=timedelta(hours=30),
    dt=timedelta(minutes=5),
)

To sample U and V as part of a larger script the following code could be used:

In [ ]:
SampleParticle = parcels.Particle.add_variable(
    [
        parcels.Variable("U", dtype=np.float32, initial=np.nan),
        parcels.Variable("V", dtype=np.float32, initial=np.nan),
    ]
)


def SampleVel_correct(particles, fieldset):
    # attention: samples particle velocity in units of the mesh (deg/s or m/s)
    particles.U, particles.V = fieldset.UV[particles]

<div class="alert alert-info">

Note that the Kernels above return the value of `U` and `V` in the units of the grid. That means that for a spherical grid, the velocities are in **degrees/s**. To convert these to **m/s**, see the [velocity conversion tutorial](./tutorial_velocityconversion.ipynb).
</div>